# 04 · Choosing the Rank

The one knob. Sweep $r$ and plot accuracy: it rises, then **plateaus** once $r$ is
big enough to hold the update's real dimensionality; higher ranks are wasted. We
overlay the subspace-capture measure from notebook 01 to show the plateau coincides
with capturing the update's true rank.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
import torch
from src.config import get_settings
from src.tasks.toy import make_toy_task, make_base_model
from src.evaluation.comparison import three_way_race
from src.analysis.subspace import captured_energy
from src.visualisation import plot_rank_sweep, save_figure
from src.results import save_json

s = get_settings()
task = make_toy_task(dim=s.toy.dim, out_dim=s.toy.out_dim, true_rank=s.toy.true_rank,
                     n_samples=s.toy.n_samples, seed=s.seed)
frozen_loss = None
rows = []
for r in s.ranks:
    base = make_base_model(task)
    res = three_way_race(base, task, r=r, alpha=s.lora.alpha, target_modules=['linear'],
                         steps=s.train.steps, lr=s.train.lr, seed=s.seed)
    frozen_loss = res['frozen'].final_loss
    lora_loss = res['lora'].final_loss
    fit = 1.0 - lora_loss / frozen_loss
    cap = captured_energy(task.delta, r)
    rows.append({'rank': r, 'lora_loss': lora_loss, 'fit_quality': fit,
                 'captured_energy': cap, 'trainable_params': res['lora'].trainable_params})
    print(f'r={r:>2}  loss={lora_loss:.3e}  fit={fit:.3f}  captured={cap:.3f}')

r= 1  loss=6.529e-01  fit=0.348  captured=0.318


r= 2  loss=3.825e-01  fit=0.618  captured=0.577


r= 4  loss=3.034e-07  fit=1.000  captured=1.000


r= 8  loss=2.175e-08  fit=1.000  captured=1.000


r=16  loss=1.255e-07  fit=1.000  captured=1.000


r=32  loss=1.862e-04  fit=1.000  captured=1.000


## Plot the sweep (PNG) and save it (JSON)

In [3]:
ranks = [row['rank'] for row in rows]
fit = [row['fit_quality'] for row in rows]
cap = [row['captured_energy'] for row in rows]
fig = plot_rank_sweep(ranks, fit, cap,
                      title=f"Accuracy plateaus at the update's true rank ({task.true_rank})")
png = save_figure(fig, '04_rank_sweep')
json_path = save_json({'true_rank': task.true_rank, 'frozen_loss': frozen_loss,
                       'sweep': rows}, '04_choosing_the_rank')
print('saved:', png.name, 'and', json_path.name)

saved: 04_rank_sweep.png and 04_choosing_the_rank.json


## Takeaway
Pick the rank from the knee, not from habit. Ranks below the update's true rank
underfit; ranks above it add parameters for nothing. `alpha` rescales the update
(effective scale $\alpha/r$), and you typically adapt the attention and feed-forward
projections rather than every layer.